In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
import glob
def compute_diversity(mols, verbose=False):
    if verbose:
        print("Computing diversity...")

    if len(mols) == 0:
        return 0

    sims = []
    fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(x), 3, 2048) for x in mols]
    for i in range(len(fps)):
        sims += DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
    return 1 - np.mean(sims)

def compute_diversity2(mols, verbose=False):
    if verbose:
        print("Computing diversity...")

    if len(mols) == 0:
        return 0

    sims = []
    fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(x), 3, 2048) for x in mols]
    sims = []
    for i in range(len(fps)):
        fps_copy = fps.copy()
        f = fps[i]
        del fps_copy[i]
        sim = DataStructs.BulkTanimotoSimilarity(f, fps_copy)
        sims.append(np.max(sim))
    return sims

def compute_novelty(mols, ref_mols):
    fps_mol = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(x), 3, 2048) for x in mols]
    fps_ref = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(x), 3, 2048) for x in ref_mols]
    sims = []
    for f in fps_mol:
        sim = DataStructs.BulkTanimotoSimilarity(f, fps_ref)
        sims.append(np.max(sim))
    return sims

def harmonic_mean(values):
    n = len(values)
    if any(v == 0 for v in values):
        return 0
    return n / sum(1/v for v in values)
def get_results(dir_path):
    results = []
    topk_df_dict = {}
    all_df_dict = {}
    for i in range(1000,30001,1000):
        path = f'{dir_path}/sampled_mols_{i}.tsv'
        try:
            df = pd.read_table(path)
            all_df_dict[i] = df
        except:
            break
        u_df = df.drop_duplicates('smiles')
        # print(f'u_df: {u_df.shape}')
        all_score_mean = u_df['reward'].mean()
        all_div = compute_diversity(u_df['smiles'])
        all_unique = df.drop_duplicates().shape[0]/df.shape[0]
        high_unique = df[df['reward']>=0.5].drop_duplicates().shape[0]
        all_hm = harmonic_mean([all_score_mean, all_div, all_unique])
        # print(f'all_score_mean: {all_score_mean}, all_div: {all_div}, all_unique: {all_unique}, all_harmonic_mean: {all_hm}')
        topk_df = u_df.nlargest(100, 'reward')
        topk_df_dict[i] = topk_df
        # print(topk_df.head(1))
        topk_score_mean = topk_df['reward'].mean()
        topk_score_median = topk_df['reward'].median()
        topk_div = compute_diversity(topk_df['smiles'])
        topk_unique = topk_df.drop_duplicates().shape[0]/topk_df.shape[0]
        topk_success = topk_df[topk_df['reward']>=0.5].shape[0]/topk_df.shape[0]
        top_hm = harmonic_mean([topk_score_mean, topk_div, all_unique])
        # print(f'topk_score_mean: {topk_score_mean}, topk_div: {topk_div}, topk_unique: {topk_unique}, topk_harmonic_mean: {top_hm}')
        results.append([i, high_unique,all_score_mean, all_div, all_unique, topk_score_mean, topk_div, topk_success, top_hm])
    results_df = pd.DataFrame(results, columns=['i', 'high_unique', 'all_score_mean', 'all_div', 'all_unique', 'topk_score_mean', 'topk_div', 'topk_success','top_hm'])
    return results_df, topk_df_dict, all_df_dict

def get_results_topk(dir_path):
    results = []
    topk_df_dict = {}
    all_df_dict = {}
    for i in range(1000,30001,1000):
        path = f'{dir_path}/sampled_mols_{i}.tsv'
        try:
            df = pd.read_table(path)
            all_df_dict[i] = df
        except:
            break
        u_df = df.drop_duplicates('smiles')
        all_unique = df.drop_duplicates().shape[0]/df.shape[0]
        high_unique = df[df['reward']>=0.5].drop_duplicates().shape[0]
        topk_df = u_df.nlargest(100, 'reward')
        topk_df_dict[i] = topk_df
        # print(topk_df.head(1))
        topk_score_mean = topk_df['reward'].mean()
        topk_score_median = topk_df['reward'].median()
        topk_div = compute_diversity(topk_df['smiles'])
        topk_unique = topk_df.drop_duplicates().shape[0]/topk_df.shape[0]
        topk_success = topk_df[topk_df['reward']>=0.5].shape[0]/topk_df.shape[0]
        top_hm = harmonic_mean([topk_score_mean, topk_div, all_unique, topk_success])
        # print(f'topk_score_mean: {topk_score_mean}, topk_div: {topk_div}, topk_unique: {topk_unique}, topk_harmonic_mean: {top_hm}')
        results.append([i, high_unique, all_unique, topk_score_mean, topk_div, topk_success, top_hm])
    results_df = pd.DataFrame(results, columns=['i', 'high_unique', 'all_unique', 'topk_score_mean', 'topk_div', 'topk_success', 'top_hm'])
    return results_df, topk_df_dict, all_df_dict

In [9]:
# GFN-Ours loss
import glob
df = pd.DataFrame()
path = glob.glob('./checkpoints/LeakGFN/seed_1/jnk3/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
display(results_df)
# path = glob.glob('./checkpoints/LeakGFN/seed_2/jnk3/*')[-1]
# results_df, topk_df_dict, all_df_dict = get_results(path)
# df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
# results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
# display(results_df)
# path = glob.glob('./checkpoints/LeakGFN/seed_3/jnk3/*')[-1]
# results_df, topk_df_dict, all_df_dict = get_results(path)
# df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
# results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
# display(results_df)
display(df)
df.T.describe()


,i,high_unique,all_score_mean,all_div,all_unique,topk_score_mean,topk_div,topk_success,top_hm
0,1000,0,0.203304,0.536537,0.112,0.2129,0.514567,0.00,0.192691
1,2000,0,0.338362,0.589885,0.818,0.4222,0.475135,0.00,0.526713
2,3000,3,0.368297,0.548557,0.975,0.4500,0.498572,0.03,0.571038
3,4000,236,0.444702,0.638193,0.872,0.5986,0.491793,1.00,0.618464


,seed_1
i,4000.000000
top_hm,0.618464
topk_score_mean,0.598600
topk_div,0.491793
all_unique,0.872000
topk_success,1.000000
high_unique,236.000000


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,1.0,1.000000,1.0000,1.000000,1.000,1.0,1.0
mean,4000.0,0.618464,0.5986,0.491793,0.872,1.0,236.0
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,4000.0,0.618464,0.5986,0.491793,0.872,1.0,236.0
25%,4000.0,0.618464,0.5986,0.491793,0.872,1.0,236.0
50%,4000.0,0.618464,0.5986,0.491793,0.872,1.0,236.0
75%,4000.0,0.618464,0.5986,0.491793,0.872,1.0,236.0
max,4000.0,0.618464,0.5986,0.491793,0.872,1.0,236.0


In [10]:
# GFN-Ours loss
import glob
df = pd.DataFrame()
path = glob.glob('./checkpoints/LeakGFN/seed_1/gsk3b/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
display(results_df)
# path = glob.glob('./checkpoints/LeakGFN/seed_2/gsk3b/*')[-1]
# results_df, topk_df_dict, all_df_dict = get_results(path)
# df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
# results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
# display(results_df)
# path = glob.glob('./checkpoints/LeakGFN/seed_3/gsk3b/*')[-1]
# results_df, topk_df_dict, all_df_dict = get_results(path)
# df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
# results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
# display(results_df)
display(df)
df.T.describe()


,i,high_unique,all_score_mean,all_div,all_unique,topk_score_mean,topk_div,topk_success,top_hm
0,1000,198,0.427516,0.750630,0.866,0.591500,0.671173,1.0,0.691997
1,2000,151,0.397165,0.831534,0.902,0.570300,0.758349,1.0,0.717572
2,3000,159,0.416031,0.808243,0.749,0.585200,0.776957,1.0,0.692680
3,4000,176,0.416913,0.804167,0.902,0.593300,0.779263,1.0,0.735763
4,5000,131,0.427421,0.811983,0.552,0.588700,0.779263,1.0,0.625845
5,6000,190,0.448138,0.839965,0.625,0.622900,0.805175,1.0,0.674557
6,7000,145,0.423993,0.793315,0.590,0.590894,0.772588,1.0,0.640804


,seed_1
i,4000.000000
top_hm,0.735763
topk_score_mean,0.593300
topk_div,0.779263
all_unique,0.902000
topk_success,1.000000
high_unique,176.000000


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,1.0,1.000000,1.0000,1.000000,1.000,1.0,1.0
mean,4000.0,0.735763,0.5933,0.779263,0.902,1.0,176.0
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,4000.0,0.735763,0.5933,0.779263,0.902,1.0,176.0
25%,4000.0,0.735763,0.5933,0.779263,0.902,1.0,176.0
50%,4000.0,0.735763,0.5933,0.779263,0.902,1.0,176.0
75%,4000.0,0.735763,0.5933,0.779263,0.902,1.0,176.0
max,4000.0,0.735763,0.5933,0.779263,0.902,1.0,176.0
